# Temperature Scaling Analysis
## Analyzing temperature-scaled Dice+CE baseline results as requested by reviewers

In [1]:
import os
import pandas as pd
import numpy as np
from tabulate import tabulate
import warnings
from scipy.stats import ttest_rel

In [2]:
# Per case metrics (raw):
TEST_METRICS_CSVS = {
    "micro_ACE": "micro_ace_raw.csv",
    "macro_ACE": "macro_ace_raw.csv",
    "micro_ECE": "micro_ece_raw.csv",
    "macro_ECE": "macro_ece_raw.csv",
    "micro_MCE": "micro_mce_raw.csv",
    "macro_MCE": "macro_mce_raw.csv",
    "DSC": "mean_dice_raw.csv",
}

# Metrics summary:
TEST_METRICS_CSVS_SUMMARY = {
    "micro_ACE": "micro_ace_summary.csv",
    "macro_ACE": "macro_ace_summary.csv",
    "micro_ECE": "micro_ece_summary.csv",
    "macro_ECE": "macro_ece_summary.csv",
    "micro_MCE": "micro_mce_summary.csv",
    "macro_MCE": "macro_mce_summary.csv",
    "DSC": "mean_dice_summary.csv",
}

SEED = 12345

In [3]:
# Temperature-scaled baseline runs
RUNS_TEMP_SCALED = {
    "acdc": "../bundles/acdc17_baseline_dice_ce_2",
    "amos": "../bundles/amos22_baseline_dice_ce_nl",
    "kits": "../bundles/kits23_baseline_dice_ce_nl",
    "brats": "../bundles/brats21_baseline_dice_ce_nl",
}

# Standard Dice+CE baseline runs (non-temp-scaled) for comparison
RUNS_DICE_CE = {
    "acdc": "../bundles/acdc17_baseline_dice_ce_2",
    "amos": "../bundles/amos22_baseline_dice_ce_nl",
    "kits": "../bundles/kits23_baseline_dice_ce_nl",
    "brats": "../bundles/brats21_baseline_dice_ce_nl",
}

In [4]:
def load_csv_files(run_path, seed_suffix=""):
    """Load CSV files for a given run path.
    
    Args:
        run_path: Path to the bundle directory
        seed_suffix: Suffix to add to seed directory (e.g., '_temp_scaled')
    """
    dataframes = {}
    
    # Load raw CSV files
    for metric, csv_file in TEST_METRICS_CSVS.items():
        csv_path = os.path.join(run_path, f'seed_{SEED}{seed_suffix}', "inference_results", csv_file)
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            if metric not in dataframes:
                dataframes[metric] = {}
            dataframes[metric]["raw"] = df
        else:
            warnings.warn(f"File does not exist: {csv_path}")
    
    # Load summary CSV files
    for metric, csv_file in TEST_METRICS_CSVS_SUMMARY.items():
        csv_path = os.path.join(run_path, f'seed_{SEED}{seed_suffix}', "inference_results", csv_file)
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            if metric not in dataframes:
                dataframes[metric] = {}
            dataframes[metric]["summary"] = df
        else:
            warnings.warn(f"File does not exist: {csv_path}")
    
    return dataframes


def load_results(runs_dict, seed_suffix=""):
    """Load all results.
    
    Args:
        runs_dict: Dictionary mapping dataset names to run paths
        seed_suffix: Suffix to add to seed directory (e.g., '_temp_scaled')
    """
    results = {}
    for dataset, run_path in runs_dict.items():
        results[dataset] = load_csv_files(run_path, seed_suffix)
        print(f"Loaded {dataset}")
    return results

In [5]:
def create_temp_scale_table(temp_results, avg_type="micro"):
    """Create a table showing temperature-scaled results for all datasets."""
    def get_metrics(dataframes, dataset_key, metric):
        try:
            df = dataframes[dataset_key][metric]["summary"]
            mean = df[df["class"] == "mean"]["mean"].values[0]
            std = df[df["class"] == "mean"]["std"].values[0]
            return mean, std
        except (KeyError, IndexError) as e:
            print(f"Error getting {metric} for {dataset_key}: {e}")
            return None, None

    table_data = []
    
    dataset_names = {
        "acdc": "ACDC 17",
        "amos": "AMOS 22",
        "kits": "KiTS 23",
        "brats": "BraTS 21"
    }
    
    for dataset in ["acdc", "amos", "kits", "brats"]:
        row = [dataset_names[dataset]]
        
        # DSC
        dsc_mean, dsc_std = get_metrics(temp_results, dataset, "DSC")
        if dsc_mean is not None:
            row.append(f"{dsc_mean:.3f} ± {dsc_std:.3f}")
        else:
            row.append("N/A")
        
        # ACE
        ace_mean, ace_std = get_metrics(temp_results, dataset, f"{avg_type}_ACE")
        if ace_mean is not None:
            row.append(f"{ace_mean:.3f} ± {ace_std:.3f}")
        else:
            row.append("N/A")
        
        # ECE
        ece_mean, ece_std = get_metrics(temp_results, dataset, f"{avg_type}_ECE")
        if ece_mean is not None:
            row.append(f"{ece_mean:.3f} ± {ece_std:.3f}")
        else:
            row.append("N/A")
        
        # MCE
        mce_mean, mce_std = get_metrics(temp_results, dataset, f"{avg_type}_MCE")
        if mce_mean is not None:
            row.append(f"{mce_mean:.3f} ± {mce_std:.3f}")
        else:
            row.append("N/A")
        
        table_data.append(row)

    headers = [
        "Dataset",
        "DSC",
        f"{avg_type.upper()} ACE",
        f"{avg_type.upper()} ECE",
        f"{avg_type.upper()} MCE",
    ]
    
    print(f"\n=== Temperature Scaling Results ({avg_type.upper()}) ===")
    print(tabulate(table_data, headers=headers, tablefmt="pipe"))

In [6]:
def calculate_temp_scale_p_values(temp_results, baseline_results):
    """Calculate p-values comparing temperature-scaled to non-temp-scaled baseline."""
    metrics_to_test = ["DSC", "macro_ACE", "macro_ECE", "macro_MCE"]
    datasets = ["acdc", "amos", "kits", "brats"]
    
    print("\n=== Statistical Significance Tests: Baseline vs Temperature Scaled (Paired t-test) ===\n")
    
    for dataset in datasets:
        print(f"\n--- {dataset.upper()} ---")
        results_table = []
        
        for metric in metrics_to_test:
            try:
                # Check if data exists
                if metric not in temp_results[dataset] or "raw" not in temp_results[dataset][metric]:
                    results_table.append([metric, "N/A"])
                    continue
                
                if metric not in baseline_results[dataset] or "raw" not in baseline_results[dataset][metric]:
                    results_table.append([metric, "N/A"])
                    continue
                
                # Load raw data (per-case values)
                df_temp = temp_results[dataset][metric]["raw"]
                df_baseline = baseline_results[dataset][metric]["raw"]
                
                # Extract mean column values
                temp_vals = df_temp["mean"].values
                baseline_vals = df_baseline["mean"].values
                
                # Perform paired t-test (baseline vs temp_scaled)
                try:
                    _, p_val = ttest_rel(baseline_vals, temp_vals)
                except Exception as e:
                    print(f"  Error in t-test for {metric}: {e}")
                    p_val = np.nan
                
                # Format p-value
                if not np.isnan(p_val):
                    p_str = f"{p_val:.3e}"
                    # Add significance markers
                    if p_val < 0.001:
                        p_str += " ***"
                    elif p_val < 0.01:
                        p_str += " **"
                    elif p_val < 0.05:
                        p_str += " *"
                else:
                    p_str = "N/A"
                
                results_table.append([metric, p_str])
                
            except Exception as e:
                print(f"  Error processing {metric}: {e}")
                results_table.append([metric, "Error"])
        
        headers = ["Metric", "Baseline vs Temp Scaled p-value"]
        print(tabulate(results_table, headers=headers, tablefmt="pipe"))
    
    print("\nSignificance levels: * p<0.05, ** p<0.01, *** p<0.001")

## Load Data

In [7]:
# Load temperature-scaled results
temp_results = load_results(RUNS_TEMP_SCALED, seed_suffix="_temp_scaled")

# Load baseline (non-temp-scaled) results for comparison
baseline_results = load_results(RUNS_DICE_CE, seed_suffix="")

Loaded acdc
Loaded amos
Loaded kits
Loaded brats
Loaded acdc
Loaded amos
Loaded kits
Loaded brats


## Macro Results

In [8]:
create_temp_scale_table(temp_results, avg_type="macro")


=== Temperature Scaling Results (MACRO) ===
| Dataset   | DSC           | MACRO ACE     | MACRO ECE     | MACRO MCE     |
|:----------|:--------------|:--------------|:--------------|:--------------|
| ACDC 17   | 0.871 ± 0.037 | 0.102 ± 0.033 | 0.001 ± 0.001 | 0.219 ± 0.067 |
| AMOS 22   | 0.882 ± 0.045 | 0.093 ± 0.019 | 0.000 ± 0.000 | 0.200 ± 0.036 |
| KiTS 23   | 0.859 ± 0.144 | 0.145 ± 0.076 | 0.002 ± 0.006 | 0.268 ± 0.130 |
| BraTS 21  | 0.905 ± 0.107 | 0.135 ± 0.061 | 0.001 ± 0.001 | 0.260 ± 0.105 |


## Micro Results

In [9]:
create_temp_scale_table(temp_results, avg_type="micro")


=== Temperature Scaling Results (MICRO) ===
| Dataset   | DSC           | MICRO ACE     | MICRO ECE     | MICRO MCE     |
|:----------|:--------------|:--------------|:--------------|:--------------|
| ACDC 17   | 0.871 ± 0.037 | 0.073 ± 0.000 | 0.001 ± 0.000 | 0.138 ± 0.000 |
| AMOS 22   | 0.882 ± 0.045 | 0.033 ± 0.000 | 0.000 ± 0.000 | 0.064 ± 0.000 |
| KiTS 23   | 0.859 ± 0.144 | 0.088 ± 0.000 | 0.001 ± 0.000 | 0.234 ± 0.000 |
| BraTS 21  | 0.905 ± 0.107 | 0.033 ± 0.000 | 0.000 ± 0.000 | 0.053 ± 0.000 |


## Statistical Significance: Baseline vs Temperature Scaled

In [10]:
calculate_temp_scale_p_values(temp_results, baseline_results)


=== Statistical Significance Tests: Baseline vs Temperature Scaled (Paired t-test) ===


--- ACDC ---
| Metric    | Baseline vs Temp Scaled p-value   |
|:----------|:----------------------------------|
| DSC       | N/A                               |
| macro_ACE | 3.322e-73 ***                     |
| macro_ECE | 3.092e-46 ***                     |
| macro_MCE | 8.031e-77 ***                     |

--- AMOS ---
| Metric    | Baseline vs Temp Scaled p-value   |
|:----------|:----------------------------------|
| DSC       | 1.082e-13 ***                     |
| macro_ACE | 7.148e-12 ***                     |
| macro_ECE | 2.930e-02 *                       |
| macro_MCE | 4.880e-19 ***                     |

--- KITS ---
| Metric    | Baseline vs Temp Scaled p-value   |
|:----------|:----------------------------------|
| DSC       | 4.260e-02 *                       |
| macro_ACE | 8.427e-21 ***                     |
| macro_ECE | 1.544e-01                         |
| macro_MCE | 4.501